# Dataset Prep v2 (Auto-detect, Convert, Merge, Split 90/10)

This notebook will:
- Auto-detect China_Drone and China_MotorBike folders under `dataset_root`
- Convert VOC XML (in `annotations/xmls`) to YOLO TXT
- Merge both datasets into `dataset_root/combined`
- Split train/val as 90%/10%
- Write `ultralytics/cfg/datasets/combined_china_motorbike.yaml`
- Run sanity checks


In [1]:
# Paths and auto-detect
from pathlib import Path
import shutil

DATASET_ROOT = Path(r"C:/Users/Christian Buenagua/Downloads/obc-yolov8/OBC-YOLOv8/ultralytics10.24/dataset_root").resolve()
DRONE = DATASET_ROOT / 'China_Drone'
MOTOR = DATASET_ROOT / 'China_MotorBike'

SRC_A_IMAGES = (DRONE / 'train' / 'images')
SRC_A_XMLS   = (DRONE / 'train' / 'annotations' / 'xmls')
SRC_B_IMAGES = (MOTOR / 'train' / 'images')
SRC_B_XMLS   = (MOTOR / 'train' / 'annotations' / 'xmls')
SRC_TEST_IMAGES = (MOTOR / 'test' / 'images') if (MOTOR / 'test' / 'images').exists() else None

TARGET_ROOT = DATASET_ROOT / 'combined'
if TARGET_ROOT.exists():
    shutil.rmtree(TARGET_ROOT)
for split in ["train","val","test"]:
    (TARGET_ROOT/split/"images").mkdir(parents=True, exist_ok=True)
    (TARGET_ROOT/split/"labels").mkdir(parents=True, exist_ok=True)

NAMES = ['D00','D10','D20','D40','Repair']
print('Dataset root:', DATASET_ROOT)
print('Target root :', TARGET_ROOT)


Dataset root: C:\Users\Christian Buenagua\Downloads\obc-yolov8\OBC-YOLOv8\ultralytics10.24\dataset_root
Target root : C:\Users\Christian Buenagua\Downloads\obc-yolov8\OBC-YOLOv8\ultralytics10.24\dataset_root\combined


In [2]:
# Conversion utils (VOC XML -> YOLO TXT)
import xml.etree.ElementTree as ET
from PIL import Image
from tqdm import tqdm

def voc_xml_to_yolo(xml_path, img_w, img_h, class_to_id):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall('object'):
        cls = obj.find('name').text
        if cls not in class_to_id:
            continue
        bbox = obj.find('bndbox')
        xmin = float(bbox.find('xmin').text)
        ymin = float(bbox.find('ymin').text)
        xmax = float(bbox.find('xmax').text)
        ymax = float(bbox.find('ymax').text)
        cx = (xmin + xmax) / 2.0 / img_w
        cy = (ymin + ymax) / 2.0 / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f"{class_to_id[cls]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    return lines

def convert_voc_dir(xml_dir: Path, images_dir: Path, out_labels_dir: Path, names: list):
    out_labels_dir.mkdir(parents=True, exist_ok=True)
    class_to_id = {n:i for i,n in enumerate(names)}
    xml_files = list(Path(xml_dir).glob('*.xml'))
    converted = 0
    for xml_path in tqdm(xml_files, desc=f"Converting VOC: {xml_dir}"):
        stem = xml_path.stem
        img_path = None
        for ext in ('.jpg','.jpeg','.png','.bmp','.tif','.tiff'):
            cand = images_dir / f"{stem}{ext}"
            if cand.exists():
                img_path = cand
                break
        if img_path is None:
            continue
        with Image.open(img_path) as im:
            w, h = im.size
        lines = voc_xml_to_yolo(xml_path, w, h, class_to_id)
        out_file = out_labels_dir / f"{stem}.txt"
        if lines:
            with open(out_file, 'w', encoding='utf-8') as f:
                f.write('\n'.join(lines))
        converted += 1
    return converted


In [3]:
# Merge and split
from tqdm import tqdm
import random
import shutil

# Convert XML -> TXT to temporary labels, then copy images and labels
TMP_A = TARGET_ROOT / 'tmp_labels_A'
TMP_B = TARGET_ROOT / 'tmp_labels_B'

print('Converting Drone XML ...')
a_conv = convert_voc_dir(SRC_A_XMLS, SRC_A_IMAGES, TMP_A, NAMES)
print('Converting Motor XML ...')
b_conv = convert_voc_dir(SRC_B_XMLS, SRC_B_IMAGES, TMP_B, NAMES)

print('Copying images+labels to train ...')

def copy_tree(src_images, src_labels, dst_images, dst_labels, exts=(".jpg",".jpeg",".png")):
    count = 0
    for img_path in tqdm(list(Path(src_images).glob('*.*'))):
        if img_path.suffix.lower() not in exts:
            continue
        stem = img_path.stem
        dst_img = Path(dst_images)/img_path.name
        dst_img.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(img_path, dst_img)
        if src_labels:
            lab = Path(src_labels)/f"{stem}.txt"
            if lab.exists():
                shutil.copy2(lab, Path(dst_labels)/lab.name)
        count += 1
    return count

train_img = TARGET_ROOT/'train'/'images'
train_lab = TARGET_ROOT/'train'/'labels'

a_count = copy_tree(SRC_A_IMAGES, TMP_A, train_img, train_lab)
b_count = copy_tree(SRC_B_IMAGES, TMP_B, train_img, train_lab)
print(f'Train copied: Drone={a_count}, Motor={b_count}')

if SRC_TEST_IMAGES:
    tcount = copy_tree(SRC_TEST_IMAGES, None, TARGET_ROOT/'test'/'images', TARGET_ROOT/'test'/'labels')
    print('Test copied:', tcount)

# Split train -> val (90/10)
all_imgs = list(train_img.glob('*.*'))
random.shuffle(all_imgs)
n_val = max(1, int(len(all_imgs)*0.1))
val_img = TARGET_ROOT/'val'/'images'
val_lab = TARGET_ROOT/'val'/'labels'
val_img.mkdir(parents=True, exist_ok=True)
val_lab.mkdir(parents=True, exist_ok=True)
for p in all_imgs[:n_val]:
    lbl = train_lab/f"{p.stem}.txt"
    shutil.move(str(p), str(val_img/p.name))
    if lbl.exists():
        shutil.move(str(lbl), str(val_lab/f"{p.stem}.txt"))
print('Split complete. Val size:', n_val)


Converting Drone XML ...


Converting VOC: C:\Users\Christian Buenagua\Downloads\obc-yolov8\OBC-YOLOv8\ultralytics10.24\dataset_root\China_Drone\train\annotations\xmls: 100%|██████████| 2401/2401 [00:43<00:00, 54.78it/s]


Converting Motor XML ...


Converting VOC: C:\Users\Christian Buenagua\Downloads\obc-yolov8\OBC-YOLOv8\ultralytics10.24\dataset_root\China_MotorBike\train\annotations\xmls: 100%|██████████| 1977/1977 [00:41<00:00, 47.52it/s]


Copying images+labels to train ...


100%|██████████| 1977/1977 [00:17<00:00, 109.89it/s]


Train copied: Drone=2401, Motor=1977


100%|██████████| 500/500 [00:03<00:00, 128.73it/s]


Test copied: 500
Split complete. Val size: 437


In [4]:
# Write combined YAML
import yaml

def write_data_yaml(root: Path, names: list, out_yaml: Path):
    data = {
        'path': str(root).replace('\\','/'),
        'train': '../train/images',
        'val': '../val/images',
        'test': '../test/images',
        'nc': len(names),
        'names': names,
    }
    out_yaml.parent.mkdir(parents=True, exist_ok=True)
    with open(out_yaml, 'w', encoding='utf-8') as f:
        yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)
    print('Wrote YAML:', out_yaml)
    print(yaml.safe_dump(data, sort_keys=False))

OUT_YAML = Path('ultralytics/cfg/datasets/combined_china_motorbike.yaml')
write_data_yaml(TARGET_ROOT, NAMES, OUT_YAML)


Wrote YAML: ultralytics\cfg\datasets\combined_china_motorbike.yaml
path: C:/Users/Christian Buenagua/Downloads/obc-yolov8/OBC-YOLOv8/ultralytics10.24/dataset_root/combined
train: ../train/images
val: ../val/images
test: ../test/images
nc: 5
names:
- D00
- D10
- D20
- D40
- Repair



In [6]:
# Delete images that have no corresponding label
from pathlib import Path

def prune_unlabeled(root: Path, splits=('train','val')):
    removed = {s: 0 for s in splits}
    for split in splits:
        img_dir = root / split / 'images'
        lbl_dir = root / split / 'labels'
        if not img_dir.exists():
            continue
        for img in list(img_dir.glob('*.*')):
            lbl = lbl_dir / f"{img.stem}.txt"
            if not lbl.exists():
                try:
                    img.unlink()
                    removed[split] += 1
                except Exception as e:
                    print(f"Could not delete {img}: {e}")
    return removed

removed_counts = prune_unlabeled(TARGET_ROOT, splits=('train','val'))
print('Removed unlabeled images:', removed_counts)


Removed unlabeled images: {'train': 5, 'val': 0}


In [7]:
# Sanity checks
from collections import Counter

def count_files(root: Path):
    counts = {}
    for split in ['train','val','test']:
        img = len(list((root/split/'images').glob('*.*')))
        lbl = len(list((root/split/'labels').glob('*.txt')))
        counts[split] = {'images': img, 'labels': lbl}
    return counts

print(count_files(TARGET_ROOT))

missing = []
for img in (TARGET_ROOT/'train'/'images').glob('*.*'):
    if not (TARGET_ROOT/'train'/'labels'/(img.stem+'.txt')).exists():
        missing.append(img.name)
print('Missing train labels:', len(missing))
if missing[:10]:
    print('Examples:', missing[:10])


{'train': {'images': 3936, 'labels': 3936}, 'val': {'images': 437, 'labels': 437}, 'test': {'images': 500, 'labels': 0}}
Missing train labels: 0
